# 04a — Feature Extraction: Hand-Crafted Acoustic Features

| Field | Detail |
|---|---|
| **Notebook** | `04a_feature_extraction_handcrafted.ipynb` |
| **Pipeline stage** | H1 — Voice Emotion Recognition - 4 of 7 (Feature Extraction, hand-crafted branch) |
| **Owner(s)** | *Thrithwaka Preethi Shakya* |
| **Created** | *28/08/2026* |
| **Last updated** | *28/08/2026* |
| **Upstream dependency** | `02_preprocessing.ipynb` (manifest + standardized audio), `03_eda.ipynb` (class-imbalance figures, used to compute class weights here) |
| **Downstream dependency** | `05b_model_cnn_lstm.ipynb`, `05c_model_ensemble.ipynb` — both train directly on this notebook's saved feature arrays |
| **Research proposal reference** | Section 5.1 (Voice Emotion Recognition — CNN-LSTM primary architecture) |
| **Literature reference** | Chowdhury, Ramanna & Kotecha (2025), *Speech emotion recognition with light weight deep neural ensemble model using hand crafted features*, Scientific Reports — source of Feature Set B's exact feature composition |

## Purpose

This notebook extracts **two distinct hand-crafted feature representations** from the standardized audio produced by `02_preprocessing.ipynb`, one for each of two of the four H1 candidate models being empirically compared (see `docs/research_proposal_mapping.md` for the full model-selection plan):

- **Feature Set A** — MFCC + pitch (F0) + energy, extracted as a **time-series sequence** (one row per audio frame). This is the proposal's primary architecture input (Section 5.1: *"Raw audio will be preprocessed using Librosa to extract MFCCs, pitch, and energy features"*), feeding the **CNN-LSTM** model in `05b_model_cnn_lstm.ipynb`. It must be a sequence, not a single averaged vector, because the LSTM component specifically needs a time axis to learn from — collapsing frames into one mean vector before the LSTM ever sees them would defeat the point of using an LSTM at all.
- **Feature Set B** — MFCC + RMSE + ZCR + Chroma STFT, extracted and **flattened into a single concatenated vector** exactly as described in Chowdhury et al. (2025)'s methodology, feeding the **CNN + CNN-BiLSTM ensemble** candidate in `05c_model_ensemble.ipynb`. This notebook deliberately replicates their described feature pipeline (squeeze ZCR/RMSE, flatten MFCC/Chroma to 1D, horizontal concatenation) so our own results on this architecture are directly comparable to their published benchmarks on the same datasets.

**A note on what this notebook does *not* do:** it does not scale or normalize the extracted features. Scaling (e.g. z-score standardization) must be fit *only* on each model's training split to avoid data leakage into validation/test — since Set A and Set B feed different models that may need different scaling strategies, that responsibility belongs in `05b`/`05c`, not here. This notebook's only responsibility is producing correct, raw, reproducible features.

**A note on an existing inconsistency to reconcile later:** `src/emotion/voice_emotion.py`'s `extract_audio_features()` function (used for live inference in the Streamlit app) currently mean-pools MFCC/pitch/energy into a single vector — a simplification used to scaffold the app before a trained model existed. Once `05b_model_cnn_lstm.ipynb` finalizes a trained CNN-LSTM expecting the full-sequence Feature Set A format defined here, that inference function will need to be updated to match. This is flagged again in the handoff section at the end of this notebook so it isn't forgotten.

## Objectives

1. Load and validate the manifest and reports produced by `02_preprocessing.ipynb` and `03_eda.ipynb`.
2. Define, with full parameter documentation, the extraction procedure for Feature Set A (sequence: MFCC + pitch + energy).
3. Define, with full parameter documentation, the extraction procedure for Feature Set B (flattened: MFCC + RMSE + ZCR + Chroma STFT).
4. Extract both feature sets for every file in the manifest, with per-file error handling and progress logging.
5. Validate that every extracted array has the exact same shape within its feature set — a hard requirement for batching fixed-size inputs into either model.
6. Diagnose and document how unvoiced/silent frames (NaN pitch estimates) are handled.
7. Compute class weights from the training split, for direct use in `05b`/`05c` (turning `03_eda.ipynb`'s imbalance finding into an actionable artifact).
8. Save both feature arrays, labels, and split assignment as a single aligned, versioned artifact, plus a metadata file documenting every parameter used.
9. Leave a structured handoff note for the two model-training notebooks that depend on this output.

## 0. Environment Setup

In [1]:
import sys
import json
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from sklearn.utils.class_weight import compute_class_weight

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):  # graceful fallback if tqdm isn't installed
        return iterable

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import settings  # noqa: E402

warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
print(f"Project root resolved to: {PROJECT_ROOT}")

Project root resolved to: C:\Users\thrit\Desktop\emotion-ai-companion-research


In [2]:
LOG_DIR = PROJECT_ROOT / "reports" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / "04a_feature_extraction_handcrafted.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, mode="w"), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("feature_extraction_handcrafted")
RANDOM_SEED = 42  # project-wide convention

## 1. Load & Validate Upstream Outputs

Confirms `02_preprocessing.ipynb` and `03_eda.ipynb` both completed and passed validation before extracting any features. Standardization parameters (sample rate, duration) are **read from notebook 02's report, not re-hardcoded here** — this guarantees this notebook can never silently drift out of sync with the actual audio it's reading.

In [3]:
preprocessing_report_path = PROJECT_ROOT / "reports" / "02_preprocessing_report.json"
eda_report_path = PROJECT_ROOT / "reports" / "03_eda_report.json"

for path in (preprocessing_report_path, eda_report_path):
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run the prerequisite notebook to completion first.")

with open(preprocessing_report_path) as f:
    preprocessing_report = json.load(f)
with open(eda_report_path) as f:
    eda_report = json.load(f)

assert preprocessing_report["class_count_validation_passed"], "Notebook 02 validation failed — fix before proceeding."
assert preprocessing_report["checksum_verification_passed"], "Notebook 02 checksum verification failed — fix before proceeding."

# Read standardization parameters from notebook 02's own report — single source of truth.
TARGET_SAMPLE_RATE = preprocessing_report["standardization"]["sample_rate_hz"]
TARGET_DURATION_SECONDS = preprocessing_report["standardization"]["duration_seconds"]
EXPECTED_N_SAMPLES = int(TARGET_SAMPLE_RATE * TARGET_DURATION_SECONDS)

logger.info(
    "Upstream validated. Standardized audio: %d Hz, %.1fs (%d samples/file).",
    TARGET_SAMPLE_RATE, TARGET_DURATION_SECONDS, EXPECTED_N_SAMPLES,
)
print(f"\u2705 Upstream notebooks validated. Target audio: {TARGET_SAMPLE_RATE} Hz, {TARGET_DURATION_SECONDS}s ({EXPECTED_N_SAMPLES} samples).")

2026-08-28 08:46:49,349 | INFO | Upstream validated. Standardized audio: 16000 Hz, 2.5s (40000 samples/file).
✅ Upstream notebooks validated. Target audio: 16000 Hz, 2.5s (40000 samples).


In [4]:
manifest_path = PROJECT_ROOT / preprocessing_report["manifest_path"]
manifest_df = pd.read_csv(manifest_path)

# Defensive fix: normalize any Windows-style backslash paths to forward slashes,
# so this notebook works correctly regardless of which OS produced the manifest.
path_columns = ["original_path", "standardized_path"]
backslash_found = False
for col in path_columns:
    if manifest_df[col].str.contains("\\\\", regex=True).any():
        backslash_found = True
        manifest_df[col] = manifest_df[col].str.replace("\\\\", "/", regex=True)

if backslash_found:
    logger.warning(
        "Backslash path separators found and normalized to forward slashes for cross-platform "
        "compatibility. Consider fixing this at the source in 02_preprocessing.ipynb (use .as_posix())."
    )

print(f"Loaded manifest: {len(manifest_df)} rows")
manifest_df.head(3)

2026-08-28 08:46:58,069 | WARNING | Backslash path separators found and normalized to forward slashes for cross-platform compatibility. Consider fixing this at the source in 02_preprocessing.ipynb (use .as_posix()).
Loaded manifest: 4068 rows


,original_path,standardized_path,source_dataset,speaker_id,gender,original_emotion_code,label,duration_sec_original,split
0,training/data/ravdess_raw/Actor_13/03-01-02-01...,data/processed/audio_standardized/ravdess/03-0...,ravdess,Actor_13,male,02,neutral,2.936271,train
1,training/data/tess_raw/TESS Toronto emotional ...,data/processed/audio_standardized/tess/YAF_dat...,tess,YAF (younger adult female),female,happy,happy,1.823011,train
2,training/data/tess_raw/TESS Toronto emotional ...,data/processed/audio_standardized/tess/OAF_rag...,tess,OAF (older adult female),female,sad,sad,2.853895,train


## 2. Label Encoding

Every model in this project must agree on the exact same integer-to-label mapping — if `05b` and `05c` each invented their own encoding independently, their outputs (and any comparison between them in `06_model_selection.ipynb`) would be meaningless. The mapping is fixed here, once, in the order defined by `config/settings.py`, and saved into this notebook's metadata output for every downstream notebook to import directly rather than redefine.

In [5]:
UNIFIED_LABELS = list(settings.emotion_labels)
label_to_idx = {label: idx for idx, label in enumerate(UNIFIED_LABELS)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

assert set(manifest_df["label"].unique()) == set(UNIFIED_LABELS), "Manifest contains labels outside the configured 6-class scheme!"

print("Label encoding (fixed for the entire project, sourced from config/settings.py):")
for label, idx in label_to_idx.items():
    print(f"  {idx}: {label}")

Label encoding (fixed for the entire project, sourced from config/settings.py):
  0: happy
  1: sad
  2: angry
  3: fear
  4: neutral
  5: surprise


## 3. Extraction Parameters

All extraction parameters are defined once, here, with the specific rationale for each — no parameter below is a default used blindly.

In [6]:
N_FFT = 2048          # standard STFT window size, matches Chowdhury et al. (2025)'s frame length
HOP_LENGTH = 512       # standard hop size; combined with N_FFT and the fixed 40,000-sample clip
                        # length, this produces an identical frame count for every file in the dataset
N_MFCC = 40             # matches both the research proposal (Section 5.1) and Chowdhury et al. (2025)
N_CHROMA = 12           # standard chromatic-scale bin count (one per pitch class), librosa default

# Pitch (F0) tracking range — librosa.pyin requires explicit fmin/fmax bounds.
# C2 (~65 Hz) to C7 (~2093 Hz) comfortably covers the full range of adult male and
# female speaking and expressive (e.g. surprised, high-pitch) voice fundamental frequency.
F0_MIN = librosa.note_to_hz("C2")
F0_MAX = librosa.note_to_hz("C7")

FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_NPZ_PATH = FEATURES_DIR / "h1_handcrafted_features.npz"
OUTPUT_METADATA_PATH = FEATURES_DIR / "h1_handcrafted_features_metadata.json"

logger.info(
    "Extraction params: n_fft=%d, hop_length=%d, n_mfcc=%d, n_chroma=%d, f0_range=[%.1f, %.1f]Hz",
    N_FFT, HOP_LENGTH, N_MFCC, N_CHROMA, F0_MIN, F0_MAX,
)

2026-08-28 08:47:18,133 | INFO | Extraction params: n_fft=2048, hop_length=512, n_mfcc=40, n_chroma=12, f0_range=[65.4, 2093.0]Hz


## 4. Feature Set A — MFCC + Pitch + Energy (Sequence)

**Used by:** `05b_model_cnn_lstm.ipynb` (proposal's primary architecture, Section 5.1)

For each standardized clip, this produces a 2D array of shape `(T, 42)`: `T` audio frames (time steps) × 42 features per frame (40 MFCC coefficients + 1 pitch estimate + 1 energy value). This shape is exactly what a CNN-LSTM expects: the CNN layers can learn local patterns across the 42 feature channels, and the LSTM layers can learn how those patterns evolve across the `T` time steps.

**Unvoiced/silent frame handling:** `librosa.pyin` returns `NaN` for frames it judges to be unvoiced (e.g. silence, or non-tonal consonant sounds). Rather than silently discard or interpolate these, we replace `NaN` with `0.0` — a standard, explicit convention meaning "no discernible pitch in this frame" — and separately report what fraction of frames this affects, so the team can judge whether it's a meaningful proportion of the data (see Section 6).

In [7]:
def extract_feature_set_a(y: np.ndarray, sr: int) -> np.ndarray:
    """Returns a (T, 42) array: [MFCC(40), pitch(1), energy(1)] per frame."""
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)  # (40, T)

    f0, _voiced_flag, _voiced_prob = librosa.pyin(
        y, fmin=F0_MIN, fmax=F0_MAX, sr=sr, frame_length=N_FFT, hop_length=HOP_LENGTH
    )  # (T,), contains NaN for unvoiced frames
    nan_fraction = float(np.isnan(f0).mean())
    f0 = np.nan_to_num(f0, nan=0.0)

    energy = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)  # (1, T)
    energy = energy.squeeze(axis=0)  # (T,)

    # Align frame counts defensively — pyin's internal padding can occasionally differ
    # by one frame from the STFT-based features; truncate to the shortest common length.
    t = min(mfcc.shape[1], len(f0), len(energy))
    mfcc, f0, energy = mfcc[:, :t], f0[:t], energy[:t]

    features = np.vstack([mfcc, f0[np.newaxis, :], energy[np.newaxis, :]]).T  # (T, 42)
    return features.astype(np.float32), nan_fraction

## 5. Feature Set B — MFCC + RMSE + ZCR + Chroma STFT (Flattened & Concatenated)

**Used by:** `05c_model_ensemble.ipynb` (CNN + CNN-BiLSTM ensemble candidate)

Replicates Chowdhury, Ramanna & Kotecha (2025)'s described feature pipeline exactly: *"The ZCR and RMSE features were computed, followed by a squeezing operation to remove singleton dimensions. MFCCs and Chroma features were also extracted and then converted into one-dimensional arrays. The extracted features were horizontally concatenated to form a single feature vector."*

This produces a single flat 1D vector per clip (not a time-series 2D array like Set A), of fixed length `D = T_zcr + T_rmse + (40 \u00d7 T_mfcc) + (12 \u00d7 T_chroma)`. Because every input clip is the same fixed length (from `02_preprocessing.ipynb`'s standardization), `D` is identical for every file — this is verified explicitly in Section 7, not assumed.

In [8]:
def extract_feature_set_b(y: np.ndarray, sr: int) -> np.ndarray:
    """Returns a flat 1D array: [ZCR, RMSE, MFCC(flattened), Chroma(flattened)], concatenated."""
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH).squeeze(axis=0)
    rmse = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH).squeeze(axis=0)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH).flatten()
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=N_CHROMA, n_fft=N_FFT, hop_length=HOP_LENGTH).flatten()

    return np.hstack([zcr, rmse, mfcc, chroma]).astype(np.float32)

## 6. Batch Extraction

Iterates every file in the manifest, in manifest row order — this order is preserved exactly in the saved output arrays, which is what lets every array (features, labels, split assignment, source dataset) stay aligned by simple row index with no separate join step required downstream.

In [9]:
feature_a_list, feature_b_list = [], []
nan_fractions = []
failed_files = []

total = len(manifest_df)
for i, row in tqdm(manifest_df.iterrows(), total=total, desc="Extracting features"):
    audio_path = PROJECT_ROOT / row["standardized_path"]
    try:
        y, sr = librosa.load(str(audio_path), sr=None)
        assert sr == TARGET_SAMPLE_RATE, f"Unexpected sample rate {sr} (expected {TARGET_SAMPLE_RATE})"
        assert len(y) == EXPECTED_N_SAMPLES, f"Unexpected sample count {len(y)} (expected {EXPECTED_N_SAMPLES})"

        feat_a, nan_frac = extract_feature_set_a(y, sr)
        feat_b = extract_feature_set_b(y, sr)

        feature_a_list.append(feat_a)
        feature_b_list.append(feat_b)
        nan_fractions.append(nan_frac)

    except Exception as exc:  # noqa: BLE001 — log and continue; failures are reported, not silently swallowed
        logger.error("Failed to extract features for %s: %s", audio_path, exc)
        failed_files.append(str(audio_path))

    if (i + 1) % 500 == 0 or (i + 1) == total:
        logger.info("Processed %d / %d files", i + 1, total)

print(f"\nExtraction complete. Succeeded: {len(feature_a_list)} / {total}. Failed: {len(failed_files)}.")
if failed_files:
    print("\u26a0\ufe0f  Failed files (see log for full tracebacks):")
    for f in failed_files[:10]:
        print(f"   {f}")

Extracting features:   0%|          | 0/4068 [00:00<?, ?it/s]

2026-08-28 08:53:07,886 | INFO | Processed 500 / 4068 files
2026-08-28 08:58:34,438 | INFO | Processed 1000 / 4068 files
2026-08-28 09:03:59,216 | INFO | Processed 1500 / 4068 files
2026-08-28 09:09:18,287 | INFO | Processed 2000 / 4068 files
2026-08-28 09:14:24,619 | INFO | Processed 2500 / 4068 files
2026-08-28 09:19:36,933 | INFO | Processed 3000 / 4068 files
2026-08-28 09:24:46,542 | INFO | Processed 3500 / 4068 files
2026-08-28 09:30:02,989 | INFO | Processed 4000 / 4068 files
2026-08-28 09:30:45,782 | INFO | Processed 4068 / 4068 files

Extraction complete. Succeeded: 4068 / 4068. Failed: 0.


> **If any files failed:** do not proceed with a silently-shrunk dataset. Investigate the logged errors — this most often means a standardized audio file is missing or was corrupted after `02_preprocessing.ipynb` ran. Re-run the relevant portion of notebook 02 if needed, rather than continuing with fewer samples than the validated manifest expects.

In [10]:
assert len(failed_files) == 0, (
    f"{len(failed_files)} files failed feature extraction — resolve before proceeding. See log: {LOG_PATH}"
)

# Keep only the successfully-processed rows, in the same order features were extracted.
manifest_df = manifest_df.iloc[: len(feature_a_list)].reset_index(drop=True)
print("\u2705 All files processed successfully with zero failures.")

✅ All files processed successfully with zero failures.


## 7. Shape Consistency Validation

Both feature sets must have **exactly the same shape for every file** — this is a hard requirement for stacking them into fixed-size batch arrays for model training. This is verified explicitly rather than assumed, even though the fixed-length standardized audio from notebook 02 should guarantee it by construction.

In [11]:
# --- Feature Set A: every (T, 42) array must share the same T ---
shapes_a = {arr.shape for arr in feature_a_list}
assert len(shapes_a) == 1, f"Feature Set A has inconsistent shapes across files: {shapes_a}"
T_A, FEATURE_DIM_A = shapes_a.pop()
assert FEATURE_DIM_A == N_MFCC + 2, f"Expected {N_MFCC + 2} features per frame, got {FEATURE_DIM_A}"
print(f"\u2705 Feature Set A: consistent shape (T={T_A}, features={FEATURE_DIM_A}) across all {len(feature_a_list)} files.")

# --- Feature Set B: every flattened vector must share the same length D ---
lengths_b = {arr.shape[0] for arr in feature_b_list}
assert len(lengths_b) == 1, f"Feature Set B has inconsistent lengths across files: {lengths_b}"
FEATURE_DIM_B = lengths_b.pop()
print(f"\u2705 Feature Set B: consistent length (D={FEATURE_DIM_B}) across all {len(feature_b_list)} files.")

✅ Feature Set A: consistent shape (T=79, features=42) across all 4068 files.
✅ Feature Set B: consistent length (D=4266) across all 4068 files.


## 8. Pitch (F0) Coverage Diagnostic

Reports what fraction of frames, on average, had no detectable pitch (and were therefore zero-filled) — useful context for interpreting Feature Set A's pitch channel later, and for catching any file/dataset with abnormally poor pitch tracking (e.g. very quiet or noisy recordings).

In [12]:
manifest_df["pitch_nan_fraction"] = nan_fractions

print("Average fraction of unvoiced/undetected-pitch frames, by source dataset:")
display(manifest_df.groupby("source_dataset")["pitch_nan_fraction"].mean().round(3))

high_nan_threshold = 0.90
high_nan_files = manifest_df[manifest_df["pitch_nan_fraction"] > high_nan_threshold]
print(f"\nFiles with >{high_nan_threshold:.0%} unvoiced frames (worth a manual listen if this list is non-trivial): {len(high_nan_files)}")
if len(high_nan_files) > 0:
    display(high_nan_files[["source_dataset", "speaker_id", "label", "pitch_nan_fraction"]].head(10))

Average fraction of unvoiced/undetected-pitch frames, by source dataset:


source_dataset
ravdess    0.427
savee      0.298
tess       0.443
Name: pitch_nan_fraction, dtype: float64


Files with >90% unvoiced frames (worth a manual listen if this list is non-trivial): 8


,source_dataset,speaker_id,label,pitch_nan_fraction
5,ravdess,Actor_21,surprise,0.924051
247,ravdess,Actor_09,angry,1.000000
645,ravdess,Actor_19,sad,0.911392
842,ravdess,Actor_05,angry,0.924051
1257,ravdess,Actor_19,angry,1.000000
1325,ravdess,Actor_21,surprise,1.000000
3499,ravdess,Actor_09,sad,0.987342
3972,ravdess,Actor_07,angry,1.000000


## 9. Compute Class Weights (Training Split Only)

`03_eda.ipynb` found a mild (~1.24x) class imbalance and recommended class weighting during training. This section computes those weights here — using **only the training split**, since weights derived from validation/test data would leak split information into the training objective — and saves them as a ready-to-use artifact so `05b`/`05c` apply the exact same, correctly-derived weights rather than each recomputing (and potentially miscalculating) their own.

In [13]:
train_labels = manifest_df.loc[manifest_df["split"] == "train", "label"]
train_labels_encoded = train_labels.map(label_to_idx).values

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(UNIFIED_LABELS)),
    y=train_labels_encoded,
)
class_weights = {idx_to_label[i]: float(w) for i, w in enumerate(class_weight_values)}

print("Computed class weights (train split only):")
for label, weight in class_weights.items():
    print(f"  {label}: {weight:.4f}")

Computed class weights (train split only):
  happy: 1.0409
  sad: 1.0390
  angry: 1.0390
  fear: 1.0390
  neutral: 0.8395
  surprise: 1.0409


## 10. Save Feature Arrays and Metadata

All arrays — both feature sets, integer-encoded labels, split assignment, and source dataset — are saved together in a single compressed `.npz` file, aligned by row index. This avoids the risk of two separately-saved files silently drifting out of alignment if one is ever regenerated independently of the other.

In [14]:
X_set_a = np.stack(feature_a_list, axis=0)  # (N, T, 42)
X_set_b = np.stack(feature_b_list, axis=0)  # (N, D)
y = manifest_df["label"].map(label_to_idx).values.astype(np.int64)
split = manifest_df["split"].values
source_dataset = manifest_df["source_dataset"].values

print(f"X_set_a shape: {X_set_a.shape}")
print(f"X_set_b shape: {X_set_b.shape}")
print(f"y shape:       {y.shape}")

np.savez_compressed(
    OUTPUT_NPZ_PATH,
    X_set_a=X_set_a,
    X_set_b=X_set_b,
    y=y,
    split=split,
    source_dataset=source_dataset,
)
logger.info("Saved feature arrays to %s", OUTPUT_NPZ_PATH)
print(f"\nSaved: {OUTPUT_NPZ_PATH} ({OUTPUT_NPZ_PATH.stat().st_size / (1024**2):.1f} MB)")

X_set_a shape: (4068, 79, 42)
X_set_b shape: (4068, 4266)
y shape:       (4068,)
2026-08-28 09:32:01,088 | INFO | Saved feature arrays to C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_handcrafted_features.npz

Saved: C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_handcrafted_features.npz (97.3 MB)


In [15]:
feature_metadata = {
    "random_seed": RANDOM_SEED,
    "source_sample_rate_hz": TARGET_SAMPLE_RATE,
    "source_duration_seconds": TARGET_DURATION_SECONDS,
    "extraction_params": {
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "n_mfcc": N_MFCC,
        "n_chroma": N_CHROMA,
        "f0_min_hz": float(F0_MIN),
        "f0_max_hz": float(F0_MAX),
    },
    "feature_set_a": {
        "description": "MFCC + pitch + energy, per-frame sequence",
        "used_by_model": "05b_model_cnn_lstm.ipynb (CNN-LSTM, proposal Section 5.1)",
        "shape": [int(T_A), int(FEATURE_DIM_A)],
        "shape_description": "(time_steps, features_per_frame)",
    },
    "feature_set_b": {
        "description": "MFCC + RMSE + ZCR + Chroma STFT, flattened and concatenated",
        "used_by_model": "05c_model_ensemble.ipynb (CNN + CNN-BiLSTM ensemble, Chowdhury et al. 2025)",
        "length": int(FEATURE_DIM_B),
        "shape_description": "(flat_vector_length,)",
    },
    "label_to_idx": label_to_idx,
    "idx_to_label": idx_to_label,
    "class_weights": class_weights,
    "total_files_processed": int(len(manifest_df)),
    "total_files_failed": int(len(failed_files)),
    "mean_pitch_nan_fraction": float(np.mean(nan_fractions)),
    "files_with_high_pitch_nan_fraction": int(len(high_nan_files)),
    "output_npz_path": str(OUTPUT_NPZ_PATH.relative_to(PROJECT_ROOT)),
    "scaling_applied": False,
    "scaling_note": "Features are UNSCALED. Fit any scaler in 05b/05c using ONLY the training split.",
}

with open(OUTPUT_METADATA_PATH, "w") as f:
    json.dump(feature_metadata, f, indent=2)

logger.info("Saved feature metadata to %s", OUTPUT_METADATA_PATH)
print(f"Saved: {OUTPUT_METADATA_PATH}")
print(json.dumps(feature_metadata, indent=2))

2026-08-28 09:32:16,814 | INFO | Saved feature metadata to C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_handcrafted_features_metadata.json
Saved: C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_handcrafted_features_metadata.json
{
  "random_seed": 42,
  "source_sample_rate_hz": 16000,
  "source_duration_seconds": 2.5,
  "extraction_params": {
    "n_fft": 2048,
    "hop_length": 512,
    "n_mfcc": 40,
    "n_chroma": 12,
    "f0_min_hz": 65.40639132514966,
    "f0_max_hz": 2093.004522404789
  },
  "feature_set_a": {
    "description": "MFCC + pitch + energy, per-frame sequence",
    "used_by_model": "05b_model_cnn_lstm.ipynb (CNN-LSTM, proposal Section 5.1)",
    "shape": [
      79,
      42
    ],
    "shape_description": "(time_steps, features_per_frame)"
  },
  "feature_set_b": {
    "description": "MFCC + RMSE + ZCR + Chroma STFT, flattened and concatenated",
    "used_by_model": "05c_model_ensemble.ipynb (CNN + CN

## 11. Final Sanity Checks

Reloads the just-saved file from disk (not the in-memory variables) — this catches any serialization bug that in-memory checks alone would miss, and confirms what's actually on disk is what a fresh notebook session (e.g. a teammate's) will actually load.

In [16]:
reloaded = np.load(OUTPUT_NPZ_PATH, allow_pickle=True)

assert reloaded["X_set_a"].shape == X_set_a.shape
assert reloaded["X_set_b"].shape == X_set_b.shape
assert reloaded["y"].shape == y.shape
assert not np.isnan(reloaded["X_set_a"]).any(), "NaN values remain in Feature Set A!"
assert not np.isnan(reloaded["X_set_b"]).any(), "NaN values remain in Feature Set B!"
assert not np.isinf(reloaded["X_set_a"]).any(), "Inf values found in Feature Set A!"
assert not np.isinf(reloaded["X_set_b"]).any(), "Inf values found in Feature Set B!"

reloaded_label_counts = pd.Series(reloaded["y"]).map(idx_to_label).value_counts()
original_label_counts = manifest_df["label"].value_counts()
assert reloaded_label_counts.sort_index().equals(original_label_counts.sort_index()), (
    "Label distribution in saved features does not match the manifest!"
)

print("\u2705 Reloaded arrays match in-memory arrays exactly.")
print("\u2705 No NaN or Inf values in either feature set.")
print("\u2705 Label distribution in saved file matches the source manifest exactly.")
print(f"\nSplit sizes in saved file: {pd.Series(reloaded['split']).value_counts().to_dict()}")

✅ Reloaded arrays match in-memory arrays exactly.
✅ No NaN or Inf values in either feature set.
✅ Label distribution in saved file matches the source manifest exactly.

Split sizes in saved file: {'train': 3254, 'val': 407, 'test': 407}


---

## Summary Note — Handoff to Next Notebooks

*(Fill in the italicized placeholders after your first real run, before committing.)*

### What was done in this notebook

1. Validated `02_preprocessing.ipynb` and `03_eda.ipynb` both passed their checks before proceeding.
2. Fixed the project-wide label-to-integer encoding, sourced from `config/settings.py`.
3. Extracted **Feature Set A** (MFCC + pitch + energy, per-frame sequence, shape `(T, 42)`) for the CNN-LSTM model.
4. Extracted **Feature Set B** (MFCC + RMSE + ZCR + Chroma STFT, flattened & concatenated, length `D`) for the CNN + CNN-BiLSTM ensemble model, replicating Chowdhury et al. (2025)'s exact feature pipeline.
5. Verified zero extraction failures, and confirmed identical output shape across every file for both feature sets.
6. Diagnosed and quantified unvoiced/undetected-pitch frame coverage.
7. Computed class weights from the training split only, ready for direct use in model training.
8. Saved everything (both feature sets, labels, split, source dataset) into one aligned `.npz` file, plus a full metadata JSON.
9. Reloaded the saved file from disk and re-verified shape, NaN/Inf-freedom, and label-distribution correctness independently of the in-memory session.

### Outputs produced by this notebook (and where to find them)

| Output | Location | Used by |
|---|---|---|
| Feature arrays (both sets, labels, split) | `data/processed/features/h1_handcrafted_features.npz` | `05b_model_cnn_lstm.ipynb` (reads `X_set_a`), `05c_model_ensemble.ipynb` (reads `X_set_b`) |
| Feature metadata (shapes, params, label mapping, class weights) | `data/processed/features/h1_handcrafted_features_metadata.json` | Both training notebooks — read shapes and class weights directly from here, don't recompute |
| Run log | `reports/logs/04a_feature_extraction_handcrafted.log` | Debugging any downstream shape-mismatch or NaN error |

### How to load this notebook's output (for `05b`/`05c`)

```python
import numpy as np
import json

data = np.load("data/processed/features/h1_handcrafted_features.npz", allow_pickle=True)
X_set_a, X_set_b, y, split = data["X_set_a"], data["X_set_b"], data["y"], data["split"]

with open("data/processed/features/h1_handcrafted_features_metadata.json") as f:
    metadata = json.load(f)
class_weights = metadata["class_weights"]  # keyed by label string — map to int index via metadata['label_to_idx']

X_train_a = X_set_a[split == "train"]
X_val_a = X_set_a[split == "val"]
X_test_a = X_set_a[split == "test"]
# ...same pattern for X_set_b, y
```

### What needs to be done next

1. **`04b_feature_extraction_raw_audio.ipynb`** (can run in parallel with this notebook — no dependency between them) — prepares normalized raw waveforms for the plain-CNN candidate and Wav2Vec2-tokenized input for the Wav2Vec2 candidate.
2. **`04c_asr_transcription.ipynb`** (can also run in parallel) — generates transcripts via ASR for the text-only DistilRoBERTa baseline used in the final H1 hypothesis test.
3. **`05b_model_cnn_lstm.ipynb`** — loads `X_set_a`, fits a scaler on the *training* portion only, trains the CNN-LSTM.
4. **`05c_model_ensemble.ipynb`** — loads `X_set_b`, fits a (possibly different) scaler on the *training* portion only, trains the CNN + CNN-BiLSTM ensemble.

**Before running `05b` or `05c`, confirm:**
- Section 7's shape-consistency checks both printed \u2705.
- Section 11's reload sanity checks all printed \u2705.
- `data/processed/features/h1_handcrafted_features_metadata.json` exists and its `total_files_failed` field reads `0`.

### Resources needed for the next step

| Resource | Needed for | Notes |
|---|---|---|
| `data/processed/features/h1_handcrafted_features.npz` | Direct model input for `05b` and `05c` | Load once per notebook; do not re-extract features independently |
| `data/processed/features/h1_handcrafted_features_metadata.json` | Input shapes for model architecture definition, class weights for the loss function | Read `feature_set_a.shape` / `feature_set_b.length` directly rather than hardcoding |
| `scikit-learn` (already in `requirements.txt`) | Fitting a `StandardScaler` on the training split inside `05b`/`05c` | No new installs required |

### Known issues / things to watch for

- **Outstanding integration task:** `src/emotion/voice_emotion.py`'s `CNNLSTMVoiceEmotionClassifier` currently mean-pools features into a single vector for live inference — this does **not** match the full-sequence `(T, 42)` format defined here. Once `05b_model_cnn_lstm.ipynb` finalizes the trained model, this inference module needs a corresponding update (extract the same per-frame sequence, not a mean-pooled vector) before the trained model can be used in the live Streamlit app. This is a deployment task, not a modeling task — track it separately so it isn't lost.
- *(Fill in anything else discovered during this run — e.g. an unusually high `pitch_nan_fraction` for a specific dataset/speaker worth a manual listen.)*

### Run metadata

- **Run by:** *Thrithwaka Preethi Shakya*
- **Date:** *28/08/2026*
- **Files processed / failed:** *Succeeded: 4068 / 4068. Failed: 0*
- **Feature Set A shape (T, 42):** *consistent shape (T=79, features=42) across all 4068 files.*
- **Feature Set B length (D):** *consistent length (D=4266) across all 4068 files.*